# Capstone Project 2: Predicting Movie Profitability

# Part 2: Data Wrangling

## 1. Introduction

In this section, we will load the "Movies Metrics" dataset, inspect its structure, handle missing values, and prepare the data for Exploratory Data Analysis (EDA). Our primary goal is to ensure the dataset is clean, consistent, and contains the necessary target variables for our profitability model.

In [22]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [23]:
# Adjust display options to ensure we can see all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Data Loading

We will load the dataset `Top Movies (Cleaned Data).csv` into a pandas DataFrame. We will then display the first few rows and the general information (columns, data types, null values) to get an initial understanding of the data structure.

In [24]:
# Load the dataset
df = pd.read_csv('Top Movies (Cleaned Data).csv')

# Display the first 5 rows
print(df.head())

# Display basic information about the dataset
print(df.info())

  id                               Movie Name Release Date  \
0  1     Star Wars Ep. VII: The Force Awakens   2015-12-16   
1  2                 Avatar: The Way of Water   2022-12-09   
2  3    Indiana Jones and the Dial of Destiny   2023-06-28   
3  4                        Avengers: Endgame   2019-04-23   
4  5  Mission: Impossible—The Final Reckoning   2025-05-21   

   Production Budget (USD)  Domestic Gross (USD)  Worldwide Gross (USD)  \
0                533200000             936662225             2056046835   
1                460000000             684075767             2315589775   
2                402300000             174480468              383963057   
3                400000000             858373000             2748242781   
4                400000000                     0                      0   

   Domestic Box Office (USD)  International Box Office (USD)  \
0                936662225.0                    1.119385e+09   
1                684075767.0                    

## 3. Data Cleaning

### 3.1 Dropping Irrelevant Columns

Based on our initial inspection, we will drop columns that are not useful for our "Profitability" prediction model.

### Columns to Drop:

- **IDs & Metadata**: `id`, `Movie URL` (No predictive value)

- **Home Media Sales**: `Est. Domestic DVD Sales (USD)`, `Est. Domestic Blu-ray Sales (USD)`, `Total Est. Domestic Video Sales (USD)`, `Video Release` (Too many missing values and irrelevant to theatrical success)

- **Granular Release Data**: `Domestic Releases`, `International Releases`, `Theater counts` (Complex text fields that are difficult to parse and often unknown pre-release)

- **Redundant/Incomplete Revenue**: `Domestic Box Office (USD)`, `International Box Office (USD)`, `Worldwide Box Office (USD)` (We will use the "Gross" columns instead, which have no missing values)

In [25]:
# List of columns to drop
cols_to_drop = [
    'id', 'Movie URL',
    'Est. Domestic DVD Sales (USD)', 'Est. Domestic Blu-ray Sales (USD)',
    'Total Est. Domestic Video Sales (USD)', 'Video Release',
    'Domestic Releases', 'International Releases', 'Theater counts',
    'Domestic Box Office (USD)', 'International Box Office (USD)', 'Worldwide Box Office (USD)'
]

# Drop the columns
df_clean = df.drop(columns=cols_to_drop)

# Verify the drop
print(f"Original shape: {df.shape}")
print(f"New shape: {df_clean.shape}")
df_clean.head(3)

Original shape: (6569, 32)
New shape: (6569, 20)


,Movie Name,Release Date,Production Budget (USD),Domestic Gross (USD),Worldwide Gross (USD),Opening Weekend (USD),Legs,Infl. Adj. Dom. BO (USD),MPAA Rating,Running Time (minutes),Franchise,Keywords,Source,Genre,Production Method,Creative Type,Production/Financing Companies,Production Countries,Languages,Domestic Share Percentage
0,Star Wars Ep. VII: The Force Awakens,2015-12-16,533200000,936662225,2056046835,247966675.0,3.78,1.191448e+09,PG-13,136.0,Star Wars,"Space Opera,Good vs. Evil,Delayed Sequel,Inter...",Original Screenplay,Adventure,"Animation,Live Action",Science Fiction,"Lucasfilm,Bad Robot",United States,English,45.6
1,Avatar: The Way of Water,2022-12-09,460000000,684075767,2315589775,134100226.0,5.10,6.935964e+08,PG-13,190.0,Avatar,"Action Adventure,Delayed Sequel,Humans as Alie...",Original Screenplay,Action,"Animation,Live Action",Science Fiction,"Lightstorm Entertainment,20th Century Studios,...",United States,English,29.5
2,Indiana Jones and the Dial of Destiny,2023-06-28,402300000,174480468,383963057,60368101.0,2.89,1.744805e+08,PG-13,142.0,Indiana Jones,"1960s,Space Program,Nazis Outside of World War...",Original Screenplay,Adventure,Live Action,Historical Fiction,"Lucasfilm,Walt Disney Pictures,Paramount Pictures",United States,English,45.4


### 3.2 Handling Missing Values

We need to address null values in `Release Date`, `MPAA Rating`, `Running Time`, `Franchise`, `Keywords`, `Production/Financing Companies`, `Production Countries`, `Languages`, `Genre`, `Source`, `Creative Type`, and `Production Method`.

### Strategies:

- **Franchise**: Null values imply the movie is not a franchise. We will convert this column to binary (1 = Franchise, 0 = Original).

- **MPAA Rating**: Missing ratings will be filled with "Unrated".

- **Running Time**: We will impute missing times with the median runtime of the dataset.

- **Release Date**: We will drop the small number of rows (~150) where the release date is unknown, as seasonality is a key feature.

- **Remaining Categorical Data**: Columns like `Production Countries` and `Languages` contain valuable information but have missing values. Since we cannot "mathematically" impute text (like calculating an average for a word), we will fill these gaps with the placeholder `"Unknown"` to preserve the rows for analysis.

In [26]:
# Handle Franchise: Convert NaN to 0, and any string value to 1
df_clean['Franchise'] = df_clean['Franchise'].notnull().astype(int)

# Handle MPAA Rating: Fill NaN with 'Unrated'
df_clean['MPAA Rating'] = df_clean['MPAA Rating'].fillna('Unrated')

# Handle Running Time: Fill NaN with Median
median_runtime = df_clean['Running Time (minutes)'].median()
df_clean['Running Time (minutes)'] = df_clean['Running Time (minutes)'].fillna(median_runtime)

# Handle Release Date: Drop rows with missing dates
df_clean = df_clean.dropna(subset=['Release Date'])

# Handle Categorical Columns
# Fill remaining categorical nulls with 'Unknown'
categorical_cols = ['Genre', 
                    'Source', 
                    'Creative Type', 
                    'Production Method',
                    'Keywords', 
                    'Production/Financing Companies', 
                    'Production Countries', 
                    'Languages']
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna('Unknown')


## 3.3 Handling Data Leakage

Columns like `Opening Weekend`, `Legs`, and `Domestic Share Percentage` are performance metrics that are only known *after* a movie is released. If we include these in our input data, our model will "cheat" by using future information to predict the past. We must drop them to ensure our model only uses **Pre-Release** information.

In [27]:
# Define columns that cause Data Leakage (Post-Release Metrics)
leakage_cols = [
    'Opening Weekend (USD)', 
    'Legs', 
    'Infl. Adj. Dom. BO (USD)', 
    'Domestic Share Percentage'
]

# Drop leakage columns
df_clean = df_clean.drop(columns=leakage_cols)

Let's do a final check on any missing values

In [28]:
# Final check of missing values
print(df_clean.isnull().sum())

Movie Name                        0
Release Date                      0
Production Budget (USD)           0
Domestic Gross (USD)              0
Worldwide Gross (USD)             0
MPAA Rating                       0
Running Time (minutes)            0
Franchise                         0
Keywords                          0
Source                            0
Genre                             0
Production Method                 0
Creative Type                     0
Production/Financing Companies    0
Production Countries              0
Languages                         0
dtype: int64


## 4. Feature Engineering

We will now create our target variables for the modeling phase.

### New Features:

- **ROI (Return on Investment)**: Calculated as Worldwide Gross / Production Budget.

- **Profitable (Target)**: A binary classification target. We define a "Success" as a movie with an ROI >= 2.5.

- **Release Year & Month**: Extracted from the Release Date to analyze seasonal trends.

In [29]:
# Create ROI Column
df_clean = df_clean[df_clean['Production Budget (USD)'] > 0] 
df_clean['ROI'] = df_clean['Worldwide Gross (USD)'] / df_clean['Production Budget (USD)']

# Create Binary Target 'Profitable'
df_clean['is_profitable'] = (df_clean['ROI'] >= 2.5).astype(int)

# Date Features
# Convert Release Date to datetime
df_clean['Release Date'] = pd.to_datetime(df_clean['Release Date'])
df_clean['Release Year'] = df_clean['Release Date'].dt.year
df_clean['Release Month'] = df_clean['Release Date'].dt.month

# Check the distribution of our new Target
print("Profitability Distribution:")
print(df_clean['is_profitable'].value_counts(normalize=True))

display(df_clean.head())

Profitability Distribution:
is_profitable
0    0.616355
1    0.383645
Name: proportion, dtype: float64


,Movie Name,Release Date,Production Budget (USD),Domestic Gross (USD),Worldwide Gross (USD),MPAA Rating,Running Time (minutes),Franchise,Keywords,Source,Genre,Production Method,Creative Type,Production/Financing Companies,Production Countries,Languages,ROI,is_profitable,Release Year,Release Month
0,Star Wars Ep. VII: The Force Awakens,2015-12-16,533200000,936662225,2056046835,PG-13,136.0,1,"Space Opera,Good vs. Evil,Delayed Sequel,Inter...",Original Screenplay,Adventure,"Animation,Live Action",Science Fiction,"Lucasfilm,Bad Robot",United States,English,3.856052,1,2015,12
1,Avatar: The Way of Water,2022-12-09,460000000,684075767,2315589775,PG-13,190.0,1,"Action Adventure,Delayed Sequel,Humans as Alie...",Original Screenplay,Action,"Animation,Live Action",Science Fiction,"Lightstorm Entertainment,20th Century Studios,...",United States,English,5.033891,1,2022,12
2,Indiana Jones and the Dial of Destiny,2023-06-28,402300000,174480468,383963057,PG-13,142.0,1,"1960s,Space Program,Nazis Outside of World War...",Original Screenplay,Adventure,Live Action,Historical Fiction,"Lucasfilm,Walt Disney Pictures,Paramount Pictures",United States,English,0.954420,0,2023,6
3,Avengers: Endgame,2019-04-23,400000000,858373000,2748242781,PG-13,181.0,1,"Ensemble,Marvel Comics,Animal Lead,Non-Chronol...",Based on Comic/Graphic Novel,Action,"Animation,Live Action",Super Hero,Marvel Studios,United States,English,6.870607,1,2019,4
4,Mission: Impossible—The Final Reckoning,2025-05-21,400000000,0,0,Unrated,105.0,0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0.000000,0,2025,5


## 5. Conclusion

We have successfully wrangled the data from a raw, messy state into a clean, analytical format.

**Summary of Actions:**

1.  **Dropped** irrelevant metadata (IDs, URLs) and home-media metrics.

2.  **Cleaned** missing values in `Runtime` (Median imputation) and `MPAA Rating` (Fill with "Unrated").

3.  **Removed** Data Leakage columns (`Opening Weekend`, `Legs`) to ensure valid predictions.

4.  **Engineered** the Target Variable: `ROI` and `is_profitable`.

The dataset is now ready for **Part 3: Exploratory Data Analysis (EDA)**.

In [30]:
# Save the cleaned dataset to a new CSV file
# Index=False ensures we don't save the row numbers as a separate column
output_filename = 'movie_data_cleaned.csv'
df_clean.to_csv(output_filename, index=False)

print(f"Success! Cleaned dataset saved to: {output_filename}")

Success! Cleaned dataset saved to: movie_data_cleaned.csv
